# Bài 8: VACUUM — Dọn dẹp file cũ & Quản lý vòng đời dữ liệu

## Mục tiêu
- Hiểu vì sao Delta Lake **không xoá file vật lý ngay** khi `DELETE`/`UPDATE`/`OPTIMIZE`/`overwrite`.
- Dùng `VACUUM` để dọn rác an toàn, hiểu tham số retention và cờ `DRY RUN`.
- Hiểu đánh đổi giữa **retention dài** (an toàn cho time travel, concurrent readers) và **chi phí lưu trữ**.


## 8.1. Vì sao cần VACUUM?

Mọi thao tác "xoá/ghi đè" trong Delta Lake (DELETE, UPDATE, MERGE, OPTIMIZE, overwrite) thực chất chỉ đánh dấu file cũ là **`remove`** (tombstone) trong `_delta_log` — **không xoá file Parquet vật lý ngay lập tức**. Lý do:

1. **Time travel**: cần file cũ còn tồn tại để đọc lại version trước.
2. **Concurrent readers**: một reader khác có thể đang giữ snapshot cũ (đang đọc dở) — xoá ngay sẽ làm query đó lỗi giữa chừng.

Hệ quả: nếu không dọn, storage sẽ phình to vô hạn theo thời gian dù dữ liệu "logic" không đổi nhiều. `VACUUM` là công cụ dọn các file **đã bị `remove` VÀ cũ hơn ngưỡng retention** (không còn thuộc bất kỳ version nào trong ngưỡng cho phép time-travel).

## 8.2. Cú pháp

```sql
VACUUM db.tbl;                          -- retention mac dinh 7 ngay (168 gio)
VACUUM db.tbl RETAIN 168 HOURS;         -- tuong minh
VACUUM db.tbl RETAIN 24 HOURS DRY RUN;  -- CHI liet ke file se bi xoa, khong xoa that
```

- **Retention mặc định: 7 ngày (168 giờ)** — con số này được chọn để lớn hơn khoảng thời gian tối đa mà 1 query đang chạy hoặc 1 concurrent transaction có thể tồn tại.
- `VACUUM` **từ chối** retention < 168 giờ trừ khi tắt safety check:
```sql
SET spark.databricks.delta.retentionDurationCheck.enabled = false;
VACUUM db.tbl RETAIN 1 HOURS;   -- chi dung khi THUC SU hieu ro rui ro (vd: moi truong test)
```
- **Luôn chạy `DRY RUN` trước** trong môi trường quan trọng để xem trước danh sách file sẽ bị xoá.

## 8.3. Đánh đổi retention

| Retention ngắn | Retention dài |
|---|---|
| Tiết kiệm chi phí lưu trữ nhanh hơn | Time travel xa hơn trong quá khứ |
| Rủi ro: query/transaction đang chạy dở có thể lỗi nếu file bị xoá giữa chừng | Tốn thêm chi phí lưu trữ file "rác" |
| Phù hợp bảng ghi rất thường xuyên (nhiều rác) | Phù hợp bảng cần audit/compliance dài hạn |

`VACUUM` **không** xoá bất cứ thứ gì trong `_delta_log` (log JSON) — đó là việc của `delta.logRetentionDuration` (mặc định 30 ngày), Delta tự dọn log cũ theo cơ chế riêng, độc lập với VACUUM data file.


## 0. Thiết lập môi trường

Notebook này chạy trong container `spark-master` (Jupyter Lab, xem `startup.sh`), kết nối tới:
- **Spark cluster**: `spark://spark-master:7077`
- **Hive Metastore**: `thrift://hive-metastore:9083` (dùng làm catalog)
- **MinIO** (S3-compatible): dữ liệu bảng managed nằm dưới `s3a://data-platform/managed/...`

Image `deltaio/delta-docker` đã cấu hình sẵn Delta Lake trong `spark-defaults.conf` nên không cần khai báo `spark.jars.packages` mỗi lần tạo `SparkSession`.

Yêu cầu: `docker compose up -d hive-metastore minio spark-master spark-worker` đã chạy trước khi mở notebook này.


In [ ]:
from pyspark.sql import SparkSession

DB_NAME = "bai08"
WAREHOUSE_DIR = "s3a://data-platform/managed"

spark = (
    SparkSession.builder
    .appName("bai08-vacuum")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "spark-master")
    .config("spark.sql.warehouse.dir", WAREHOUSE_DIR)
    .enableHiveSupport()
    .getOrCreate()
)

spark.sql(f"CREATE DATABASE IF NOT EXISTS {DB_NAME} LOCATION '{WAREHOUSE_DIR}/{DB_NAME}.db'")
spark.sql(f"USE {DB_NAME}")
spark


## 8.4. Ví dụ minh hoạ

In [ ]:
spark.sql("DROP TABLE IF EXISTS bai08.txn")
spark.createDataFrame([(1,100.0),(2,200.0),(3,300.0)], ["id","amount"]).write.format("delta").saveAsTable("bai08.txn")

spark.sql("UPDATE bai08.txn SET amount = amount * 2 WHERE id = 1")   # tao file remove + add moi
spark.sql("DELETE FROM bai08.txn WHERE id = 2")                       # tao file remove

loc = spark.sql("DESCRIBE EXTENDED bai08.txn").filter("col_name = 'Location'").collect()[0]["data_type"]
p = sc._jvm.org.apache.hadoop.fs.Path(loc)
fs5 = p.getFileSystem(sc._jsc.hadoopConfiguration())
print("File Parquet vat ly con tren storage (bao gom ca file da 'remove'):",
      len([f for f in fs5.listStatus(p) if f.getPath().getName().endswith(".parquet")]))


In [ ]:
# Retention mac dinh 168h -> chua co gi du dieu kien de xoa. Ha retention xuong 0h chi de MINH HOA trong bai hoc.
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")

dry_run = spark.sql("VACUUM bai08.txn RETAIN 0 HOURS DRY RUN")
dry_run.show(truncate=False)
print("So file SE bi xoa (dry run, chua xoa that):", dry_run.count())


In [ ]:
spark.sql("VACUUM bai08.txn RETAIN 0 HOURS")

n_after = len([f for f in fs5.listStatus(p) if f.getPath().getName().endswith(".parquet")])
print("File Parquet con lai sau VACUUM:", n_after)

spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "true")  # bat lai an toan mac dinh


In [ ]:
# Time travel ve version truoc UPDATE/DELETE gio se LOI vi file da bi VACUUM xoa that
try:
    spark.sql("SELECT * FROM bai08.txn VERSION AS OF 0").show()
except Exception as e:
    print("Loi nhu du kien - file cua version 0 da bi VACUUM xoa:", type(e).__name__)


## 8.5. Thực hành

> ⚠️ Bài học này hạ retention xuống rất thấp **chỉ để quan sát hiệu ứng trong 1 buổi học**. Trong thực tế **không hạ `retentionDurationCheck`** trừ khi bạn chắc chắn không có transaction/reader nào khác đang chạy.

**Bài 1** — Tạo bảng `bai08.sessions`, thực hiện `UPDATE` và `DELETE` để sinh ra file "rác" (đã bị remove trong log nhưng còn vật lý).

**Bài 2** — Chạy `VACUUM ... DRY RUN` với retention mặc định (168 giờ) — quan sát: có file nào bị liệt kê không? Vì sao?

**Bài 3** — Hạ retention xuống `0 HOURS` (tắt safety check), chạy `DRY RUN` trước, rồi chạy VACUUM thật. Đếm số file trước/sau.

**Bài 4** — Sau khi VACUUM ở Bài 3, thử `SELECT ... VERSION AS OF 0` trên bảng đó — xác nhận time travel về version cũ bị lỗi.

**Bài 5 (tư duy)** — Một bảng Delta được 1 job streaming ghi liên tục (mỗi vài giây append 1 lần) và cũng được `MERGE` cập nhật mỗi giờ. Đề xuất một lịch chạy `OPTIMIZE` và `VACUUM` hợp lý cho bảng này (tần suất, retention), giải thích ngắn gọn lý do.


### Vùng làm bài — Bài 1

In [ ]:
# TODO: Bài 1


### Vùng làm bài — Bài 2

In [ ]:
# TODO: Bài 2


### Vùng làm bài — Bài 3

In [ ]:
# TODO: Bài 3


### Vùng làm bài — Bài 4

In [ ]:
# TODO: Bài 4


### Vùng làm bài — Bài 5

_Viết câu trả lời của bạn ở đây._

---
## Gợi ý / đáp án tham khảo

In [ ]:
# Dap an Bai 1
spark.sql("DROP TABLE IF EXISTS bai08.sessions")
spark.createDataFrame([(1,"active"),(2,"active"),(3,"active")], ["id","status"]).write.format("delta").saveAsTable("bai08.sessions")
spark.sql("UPDATE bai08.sessions SET status = 'closed' WHERE id = 1")
spark.sql("DELETE FROM bai08.sessions WHERE id = 2")


In [ ]:
# Dap an Bai 2
dr = spark.sql("VACUUM bai08.sessions DRY RUN")   # retention mac dinh 168h
dr.show(truncate=False)
print("So file du kien xoa:", dr.count(), "-> thuong la 0 vi file con qua moi (chua qua 168h)")


In [ ]:
# Dap an Bai 3
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")
loc3 = spark.sql("DESCRIBE EXTENDED bai08.sessions").filter("col_name = 'Location'").collect()[0]["data_type"]
p3 = sc._jvm.org.apache.hadoop.fs.Path(loc3)
fs6 = p3.getFileSystem(sc._jsc.hadoopConfiguration())
print("Truoc:", len([f for f in fs6.listStatus(p3) if f.getPath().getName().endswith(".parquet")]))

spark.sql("VACUUM bai08.sessions RETAIN 0 HOURS DRY RUN").show(truncate=False)
spark.sql("VACUUM bai08.sessions RETAIN 0 HOURS")

print("Sau:", len([f for f in fs6.listStatus(p3) if f.getPath().getName().endswith(".parquet")]))
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "true")


In [ ]:
# Dap an Bai 4
try:
    spark.sql("SELECT * FROM bai08.sessions VERSION AS OF 0").show()
except Exception as e:
    print("Loi nhu du kien:", type(e).__name__)


**Đáp án Bài 5**: Chạy `OPTIMIZE` **1 lần/ngày** (ví dụ vào giờ thấp điểm) — không nên chạy sau mỗi micro-batch streaming vì chi phí compact liên tục sẽ lớn hơn lợi ích, và các file mới ghi trong ngày vẫn còn "hợp lý" về số lượng cho tới lần compact tiếp theo. `VACUUM` nên chạy **1 lần/ngày hoặc vài ngày/lần** với retention **giữ nguyên mặc định 7 ngày (168 giờ)** — đủ dài để bất kỳ query đang chạy dở, hoặc nhu cầu debug/rollback trong tuần, vẫn dùng time travel được, đồng thời vẫn dọn được rác đều đặn vì có ingest liên tục sinh nhiều file. Không hạ retention xuống thấp cho bảng production dạng này.
